# HPC CPU Training — Mixed-Autonomy Ring Road RL

Adapted from `colab_gpu_training.ipynb` for **CPU-only HPC** via Open OnDemand (SLURM).

### Key differences from Colab GPU version
| | Colab (GPU) | HPC (CPU) |
|---|---|---|
| Device | CUDA (T4/V100) | CPU (16 cores) |
| Parallelism | GPU batched tensors | Multi-threaded MKL/OpenMP |
| NUM_ENVS | 1024 | 256 (smaller batches, better cache) |
| MINIBATCH_SIZE | 4096 | 2048 |
| Auto-tune | Serial trials | 4 parallel trials × 4 threads |
| Storage | Google Drive | Local filesystem |
| Throughput | ~165k steps/s | ~15-25k steps/s (16 cores) |

### SLURM configuration
```
Account:    cea_seism
Partition:  cpu (16 tasks per node)
Walltime:   08:00:00  (training ~1h, auto-tune ~5-6h)
Nodes:      1
CPUs/task:  16
```

---
## 1. Environment Setup

In [ ]:
# Install dependencies (run once)
# If using a conda/venv environment, install these beforehand:
#   pip install torch numpy matplotlib pyyaml pandas optuna

import os, sys

# Ensure the repo root is on the path
REPO_ROOT = os.getcwd()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

assert os.path.isfile('src/gpu/vec_env.py'), (
    f"Missing source files. Current dir: {os.getcwd()}\n"
    "Launch this notebook from the repository root."
)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for HPC
import matplotlib.pyplot as plt
import time
import json
from pathlib import Path

# --- CPU configuration ---
NUM_CPU_CORES = int(os.environ.get('SLURM_CPUS_PER_TASK', '16'))
torch.set_num_threads(NUM_CPU_CORES)
torch.set_num_interop_threads(2)  # inter-op parallelism (small)

DEVICE = torch.device('cpu')

print(f"PyTorch: {torch.__version__}")
print(f"CPU threads: {torch.get_num_threads()} (detected {NUM_CPU_CORES} SLURM cores)")
print(f"Device: {DEVICE}")

---
## 2. Hyperparameters

Tuned for CPU: smaller batch sizes, fewer parallel envs.
Physics, reward weights, and PPO algorithm are identical to the GPU version.

In [ ]:
# =================== HYPERPARAMETERS ===================

# --- Parallelism (CPU-tuned) ---
# Smaller than GPU (1024) — CPU benefits from fitting in L2/L3 cache.
# With 16 threads, 256 envs gives good throughput without cache thrashing.
NUM_ENVS = 256

# --- Ring-road physics ---
N_VEHICLES = 22
RING_LENGTH = 300.0
DT = 0.1
EPISODE_STEPS = 1500        # 150 seconds per episode

# --- PPO ---
TOTAL_TIMESTEPS = 5_000_000
ROLLOUT_STEPS = 128
PPO_EPOCHS = 10
MINIBATCH_SIZE = 2048       # Smaller than GPU (4096) for CPU cache efficiency
GAMMA = 0.99
LAM_GAE = 0.95
EPS_CLIP = 0.2
LR_ACTOR = 3e-4
LR_CRITIC = 1e-3
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
MAX_GRAD_NORM = 0.5
CLIP_VALUE_LOSS = True

# --- Network ---
HIDDEN_DIM = 128
NUM_HIDDEN = 2
DELTA_ALPHA_MAX = 0.5

# --- Domain randomisation ---
HR_OPTIONS = (0.0, 0.25, 0.5, 0.75)
HR_WEIGHTS = (0.15, 0.25, 0.30, 0.30)

# --- Physics tuning ---
CTH_D0 = 3.5
V_REF = 5.5

# --- Safety thresholds ---
S_MIN_REWARD = 2.0
TAU_MIN_REWARD = 0.8

# --- Reward weights ---
W_V = 5.0
W_J = 0.15
W_SS = 0.75
W_ALPHA = 0.3
W_ALPHA_FINAL = 0.10         # Gradient guide floor (prevents alpha>1 drift)
W_SIGMA = 0.5
W_SIGMA_CAV = 2.0
FLEET_PENALTY_SCALING = True

# --- Damping reward ---
W_DAMP = 0.3
SIGMA_REF_DAMP = 1.0

# --- Adaptive v_ref ---
ADAPTIVE_V_REF = True
V_REF_DELTA = 3.0

# --- Perturbation curriculum ---
PERTURB_CURRICULUM = True
PERTURB_DV_MIN = -4.0
PERTURB_DV_MAX = -1.0
PERTURB_TIME_MIN = 2.0
PERTURB_TIME_MAX = 8.0
PERTURB_RANDOM_TARGET = True

# --- Normaliser ---
NORMALIZER_WARMUP = 50_000

# --- Logging & checkpoints ---
LOG_INTERVAL = 5
EVAL_INTERVAL = 20
SAVE_INTERVAL = 50
CHECKPOINT_DIR = 'output/hpc_train'

SEED = 42

# Computed
STEPS_PER_ROLLOUT = NUM_ENVS * ROLLOUT_STEPS
NUM_UPDATES = TOTAL_TIMESTEPS // STEPS_PER_ROLLOUT

print(f"Parallel envs: {NUM_ENVS}, rollout length: {ROLLOUT_STEPS}")
print(f"Training plan: {NUM_UPDATES} PPO updates, {STEPS_PER_ROLLOUT:,} steps/update, {TOTAL_TIMESTEPS:,} total")
print(f"CPU threads: {torch.get_num_threads()}")

---
## 2b. Automated Reward-Weight Tuning (Optional)

Bayesian hyperparameter search using Optuna. On CPU with 16 cores,
runs **4 parallel trials** (4 threads each) for full core utilisation.

Set `RUN_AUTO_TUNE = True` to search for optimal weights before training.
The best configuration is applied automatically.

In [ ]:
# =================== AUTO-TUNE (OPTIONAL) ===================
RUN_AUTO_TUNE = False          # Toggle this
N_TUNE_TRIALS = 40             # 30-50 recommended
SEARCH_TIMESTEPS = 2_000_000   # 2M steps per trial

# CPU parallelism: 4 parallel trials × 4 threads = 16 cores fully used
N_JOBS = 4
THREADS_PER_TRIAL = NUM_CPU_CORES // N_JOBS  # 4

if RUN_AUTO_TUNE:
    import optuna
    from src.gpu.auto_tune import run_study, best_params_as_code

    study = run_study(
        device=DEVICE,
        n_trials=N_TUNE_TRIALS,
        search_timesteps=SEARCH_TIMESTEPS,
        num_envs=NUM_ENVS,
        seed=SEED,
        n_jobs=N_JOBS,
        threads_per_trial=THREADS_PER_TRIAL,
    )

    # Apply best params
    bp = study.best_params
    W_V = bp['w_v']
    W_J = bp['w_j']
    W_SS = bp['w_ss']
    W_SIGMA = bp['w_sigma']
    W_SIGMA_CAV = bp['w_sigma_cav']
    W_DAMP = bp['w_damp']
    W_ALPHA_FINAL = bp['w_alpha_final']
    V_REF_DELTA = bp['v_ref_delta']
    HIDDEN_DIM = bp['hidden_dim']

    # Restore full thread count for training
    torch.set_num_threads(NUM_CPU_CORES)

    print('\n\u2713 Best params applied. Proceed to training.')
    print(best_params_as_code(study))
else:
    print('Auto-tune skipped. Using manually-set hyperparameters.')

---
## 3. Initialise Environment & Trainer

In [ ]:
from src.gpu.vec_env import VecRingRoadEnv, VecEnvConfig
from src.gpu.gpu_ppo import GPUPPOTrainer, GPURunningNormalizer
from src.agents.rl_types import OBS_DIM
import dataclasses

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Build vectorised environment
env_cfg = VecEnvConfig(
    N=N_VEHICLES, L=RING_LENGTH, dt=DT, episode_steps=EPISODE_STEPS,
    hr_options=HR_OPTIONS,
    hr_weights=HR_WEIGHTS,
    cth_d0=CTH_D0,
    v_ref=V_REF,
    s_min_reward=S_MIN_REWARD,
    tau_min_reward=TAU_MIN_REWARD,
    w_v=W_V, w_j=W_J, w_ss=W_SS,
    w_alpha=W_ALPHA, w_alpha_final=W_ALPHA_FINAL,
    w_sigma=W_SIGMA, w_sigma_cav=W_SIGMA_CAV,
    fleet_penalty_scaling=FLEET_PENALTY_SCALING,
    w_damp=W_DAMP, sigma_ref_damp=SIGMA_REF_DAMP,
    adaptive_v_ref=ADAPTIVE_V_REF, v_ref_delta=V_REF_DELTA,
    perturb_curriculum=PERTURB_CURRICULUM,
    perturb_dv_min=PERTURB_DV_MIN, perturb_dv_max=PERTURB_DV_MAX,
    perturb_time_min=PERTURB_TIME_MIN, perturb_time_max=PERTURB_TIME_MAX,
    perturb_random_target=PERTURB_RANDOM_TARGET,
)
env = VecRingRoadEnv(num_envs=NUM_ENVS, cfg=env_cfg, device=DEVICE)

def _make_eval_cfg(env_cfg, hr):
    overrides = dataclasses.asdict(env_cfg)
    overrides['hr_options'] = (hr,)
    overrides['hr_weights'] = None
    overrides['perturb_curriculum'] = False
    return VecEnvConfig(**overrides)

trainer = GPUPPOTrainer(
    obs_dim=OBS_DIM, hidden_dim=HIDDEN_DIM, num_hidden=NUM_HIDDEN,
    eps_clip=EPS_CLIP, clip_value_loss=CLIP_VALUE_LOSS,
    entropy_coef=ENTROPY_COEF,
    value_coef=VALUE_COEF, max_grad_norm=MAX_GRAD_NORM,
    ppo_epochs=PPO_EPOCHS, minibatch_size=MINIBATCH_SIZE,
    gamma=GAMMA, lam_gae=LAM_GAE,
    rollout_steps=ROLLOUT_STEPS, num_envs=NUM_ENVS,
    N=N_VEHICLES, device=DEVICE,
    normalizer_warmup=NORMALIZER_WARMUP,
)

env.reset()

# Sanity check
dummy_actions = torch.zeros(NUM_ENVS, N_VEHICLES, device=DEVICE)
obs, reward, done, mask, info = env.step(dummy_actions)
print(f"Obs shape: {obs.shape}, Reward shape: {reward.shape}")
print(f"OBS_DIM = {OBS_DIM}")
print(f"\n\u2713 Environment and trainer initialised on CPU.")

---
## 4. Training Loop

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

log_rewards = []
log_pg_loss = []
log_v_loss = []
log_entropy = []
log_alpha_mean = []
log_steps = []
log_fps = []

env.reset()
global_step = trainer.global_step

sort_idx, gaps_init, dv_init = env._sort_and_gaps()
mu_v_init, sigma_v_sq_init = env._upstream_stats(sort_idx)
a_leader_init = env.a_prev.gather(1, env._leader_idx)
obs, mask = env.build_obs(mu_v_init, sigma_v_sq_init, gaps_init, dv_init, a_leader_init)

t_start = time.time()
best_eval_reward = -float('inf')

print(f"Starting training from step {global_step:,}")
print(f"Target: {TOTAL_TIMESTEPS:,} total steps, {NUM_UPDATES} PPO updates")
print(f"w_alpha annealing: {W_ALPHA:.3f} \u2192 {W_ALPHA_FINAL:.3f}")
print(f"CPU threads: {torch.get_num_threads()}")
print('=' * 70)

for update_idx in range(1, NUM_UPDATES + 1):
    progress = global_step / TOTAL_TIMESTEPS
    trainer.update_lr(progress)
    env.current_w_alpha = W_ALPHA + (W_ALPHA_FINAL - W_ALPHA) * progress

    if global_step >= NORMALIZER_WARMUP and not trainer.normalizer.frozen:
        trainer.normalizer.freeze()
        print(f"[Step {global_step:,}] Normaliser frozen.")

    rollout_rewards = []
    rollout_alphas = []
    trainer.buffer.reset()

    for step_t in range(ROLLOUT_STEPS):
        obs_norm = trainer.normalizer.normalize(obs)

        if not trainer.normalizer.frozen:
            cav_obs = obs[mask]
            if cav_obs.shape[0] > 0:
                trainer.normalizer.update_batch(cav_obs)

        actions, log_probs, values = trainer.select_actions(obs_norm, mask)
        delta_alphas = actions.squeeze(2)
        next_obs, reward, done, next_mask, info = env.step(delta_alphas)

        trainer.buffer.add(
            obs=obs_norm, actions=actions, log_probs=log_probs,
            rewards=reward, values=values, dones=done, masks=mask
        )

        rollout_rewards.append(reward.mean().item())
        if info.get('alpha_mean') is not None:
            rollout_alphas.append(
                info['alpha_mean'].item() if isinstance(info['alpha_mean'], torch.Tensor)
                else info['alpha_mean']
            )

        if done.any():
            env.auto_reset(done)
            sort_idx_r, gaps_r, dv_r = env._sort_and_gaps()
            mu_v_r, sig_r = env._upstream_stats(sort_idx_r)
            a_leader_r = env.a_prev.gather(1, env._leader_idx)
            fresh_obs, fresh_mask = env.build_obs(mu_v_r, sig_r, gaps_r, dv_r, a_leader_r)
            reset_mask = done.unsqueeze(1).unsqueeze(2).expand_as(next_obs)
            next_obs = torch.where(reset_mask, fresh_obs, next_obs)
            reset_mask2 = done.unsqueeze(1).expand_as(next_mask)
            next_mask = torch.where(reset_mask2, fresh_mask, next_mask)

        obs = next_obs
        mask = next_mask
        global_step += NUM_ENVS

    trainer.global_step = global_step
    loss_info = trainer.update()

    mean_reward = np.mean(rollout_rewards)
    mean_alpha = np.mean(rollout_alphas) if rollout_alphas else 0.0
    elapsed = time.time() - t_start
    fps = global_step / max(elapsed, 1)

    log_rewards.append(mean_reward)
    log_pg_loss.append(loss_info['pg_loss'])
    log_v_loss.append(loss_info['v_loss'])
    log_entropy.append(loss_info['entropy'])
    log_alpha_mean.append(mean_alpha)
    log_steps.append(global_step)
    log_fps.append(fps)

    if update_idx % LOG_INTERVAL == 0:
        lr_now = trainer.optimizer.param_groups[0]['lr']
        print(
            f"[Update {update_idx:>4}/{NUM_UPDATES}]  "
            f"Step {global_step:>9,}  "
            f"R={mean_reward:+.3f}  "
            f"PG={loss_info['pg_loss']:.4f}  "
            f"VL={loss_info['v_loss']:.4f}  "
            f"H={loss_info['entropy']:.3f}  "
            f"\u03b1={mean_alpha:.3f}  "
            f"w\u03b1={env.current_w_alpha:.3f}  "
            f"LR={lr_now:.2e}  "
            f"FPS={fps:,.0f}"
        )

    if update_idx % SAVE_INTERVAL == 0:
        ckpt_name = f'ckpt_{global_step}.pt'
        trainer.save(os.path.join(CHECKPOINT_DIR, ckpt_name))
        trainer.save(os.path.join(CHECKPOINT_DIR, 'ckpt_latest.pt'))
        print(f"  \u21b3 Checkpoint saved: {ckpt_name}")

# Final save
trainer.save(os.path.join(CHECKPOINT_DIR, 'ckpt_final.pt'))
total_time = time.time() - t_start
print('=' * 70)
print(f"Training complete! {global_step:,} steps in {total_time/60:.1f} min ({global_step/total_time:,.0f} steps/sec)")

---
## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
ax.plot(log_steps, log_rewards, alpha=0.3, color='blue')
window = max(len(log_rewards) // 50, 5)
if len(log_rewards) > window:
    smooth = np.convolve(log_rewards, np.ones(window)/window, mode='valid')
    ax.plot(log_steps[window-1:], smooth, color='blue', linewidth=2)
ax.set_xlabel('Global Steps'); ax.set_ylabel('Mean Reward')
ax.set_title('Episode Reward'); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(log_steps, log_pg_loss, alpha=0.5, color='red')
ax.set_xlabel('Global Steps'); ax.set_ylabel('Policy Loss')
ax.set_title('Policy Gradient Loss'); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.plot(log_steps, log_v_loss, alpha=0.5, color='green')
ax.set_xlabel('Global Steps'); ax.set_ylabel('Value Loss')
ax.set_title('Value Function Loss'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(log_steps, log_entropy, alpha=0.5, color='purple')
ax.set_xlabel('Global Steps'); ax.set_ylabel('Entropy')
ax.set_title('Policy Entropy'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(log_steps, log_alpha_mean, alpha=0.5, color='orange')
ax.set_xlabel('Global Steps'); ax.set_ylabel('Mean \u03b1')
ax.set_title('Average Headway Scale (\u03b1)'); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
ax.plot(log_steps, log_fps, alpha=0.5, color='teal')
ax.set_xlabel('Global Steps'); ax.set_ylabel('Steps / sec')
ax.set_title('Training Throughput'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150)
plt.show()
print(f"Final reward: {log_rewards[-1]:+.4f}, Best reward: {max(log_rewards):+.4f}")

---
## 6. Evaluation Sweep

In [ ]:
@torch.no_grad()
def sweep_evaluation(trainer, env_cfg, device, num_eval_envs=16, num_seeds=5):
    """Run full sweep: 3 modes \u00d7 4 HRs \u00d7 num_seeds."""
    hr_values = [0.0, 0.25, 0.5, 0.75]
    modes = ['baseline', 'adaptive', 'rl']
    results = []

    for hr in hr_values:
        for mode in modes:
            for seed in range(num_seeds):
                torch.manual_seed(seed)

                eval_cfg = _make_eval_cfg(env_cfg, hr)
                eval_env = VecRingRoadEnv(
                    num_envs=num_eval_envs, cfg=eval_cfg, device=device
                )
                eval_env.reset()

                si, gi, di = eval_env._sort_and_gaps()
                mu, sig = eval_env._upstream_stats(si)
                al = eval_env.a_prev.gather(1, eval_env._leader_idx)
                obs, mask = eval_env.build_obs(mu, sig, gi, di, al)

                total_rewards = torch.zeros(num_eval_envs, device=device)
                v_trace = []

                for step in range(eval_cfg.episode_steps):
                    if mode == 'baseline':
                        delta_alphas = torch.zeros(num_eval_envs, eval_cfg.N, device=device)
                        eval_env.meso_rho.zero_()
                        eval_env.meso_alpha_prev.fill_(1.0)
                    elif mode == 'adaptive':
                        delta_alphas = torch.zeros(num_eval_envs, eval_cfg.N, device=device)
                    elif mode == 'rl':
                        obs_norm = trainer.normalizer.normalize(obs)
                        actions, _, _ = trainer.select_actions(obs_norm, mask, deterministic=True)
                        delta_alphas = actions.squeeze(2)

                    obs, reward, done, mask, info = eval_env.step(delta_alphas)
                    total_rewards += reward
                    v_trace.append(eval_env.v.mean().item())

                mean_r = total_rewards.mean().item()
                final_v = np.mean(v_trace[-100:])
                results.append({
                    'hr': hr, 'mode': mode, 'seed': seed,
                    'reward': mean_r, 'final_speed': final_v,
                    'mean_speed': np.mean(v_trace),
                })

            print(f"  HR={hr:.2f} | {mode:>8s} | "
                  f"R={np.mean([r['reward'] for r in results if r['hr']==hr and r['mode']==mode]):+.3f} \u00b1 "
                  f"{np.std([r['reward'] for r in results if r['hr']==hr and r['mode']==mode]):.3f}")

    return results

print('Running experiment sweep (3 modes \u00d7 4 HRs \u00d7 5 seeds = 60 episodes)...')
sweep_results = sweep_evaluation(trainer, env_cfg, DEVICE)
print('Done!')

---
## 7. Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(sweep_results)

summary = df.groupby(['hr', 'mode']).agg(
    reward_mean=('reward', 'mean'),
    reward_std=('reward', 'std'),
    speed_mean=('mean_speed', 'mean'),
    final_speed_mean=('final_speed', 'mean'),
).round(3)
print(summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, title in zip(
    axes,
    ['reward', 'mean_speed', 'final_speed'],
    ['Total Reward', 'Mean Speed (m/s)', 'Final Speed (m/s)']
):
    for mode, color in [('baseline', 'gray'), ('adaptive', 'blue'), ('rl', 'green')]:
        sub = df[df['mode'] == mode]
        means = sub.groupby('hr')[metric].mean()
        stds = sub.groupby('hr')[metric].std()
        ax.errorbar(means.index, means.values, yerr=stds.values,
                    label=mode, marker='o', capsize=5, color=color)
    ax.set_xlabel('Human Rate')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'sweep_comparison.png'), dpi=150)
plt.show()

---
## 8. Space-Time Diagrams

In [ ]:
@torch.no_grad()
def record_episode(trainer, env_cfg, hr, device, mode='rl'):
    eval_cfg = _make_eval_cfg(env_cfg, hr)
    eval_env = VecRingRoadEnv(num_envs=1, cfg=eval_cfg, device=device)
    eval_env.reset()

    positions = []
    speeds = []
    is_cav = eval_env.is_cav[0].cpu().numpy()

    si, gi, di = eval_env._sort_and_gaps()
    mu, sig = eval_env._upstream_stats(si)
    al = eval_env.a_prev.gather(1, eval_env._leader_idx)
    obs, mask = eval_env.build_obs(mu, sig, gi, di, al)

    for step in range(eval_cfg.episode_steps):
        positions.append(eval_env.x[0].cpu().numpy().copy())
        speeds.append(eval_env.v[0].cpu().numpy().copy())

        if mode == 'baseline':
            da = torch.zeros(1, eval_cfg.N, device=device)
            eval_env.meso_rho.zero_()
            eval_env.meso_alpha_prev.fill_(1.0)
        elif mode == 'adaptive':
            da = torch.zeros(1, eval_cfg.N, device=device)
        else:
            obs_n = trainer.normalizer.normalize(obs)
            acts, _, _ = trainer.select_actions(obs_n, mask, deterministic=True)
            da = acts.squeeze(2)

        obs, _, _, mask, _ = eval_env.step(da)

    return np.array(positions), np.array(speeds), is_cav


for mode_name in ['baseline', 'adaptive', 'rl']:
    pos, spd, is_cav_arr = record_episode(trainer, env_cfg, hr=0.75, device=DEVICE, mode=mode_name)

    fig, ax = plt.subplots(figsize=(14, 6))
    time_arr = np.arange(pos.shape[0]) * DT

    for veh_idx in range(N_VEHICLES):
        color = 'blue' if is_cav_arr[veh_idx] else 'gray'
        alpha_val = 0.8 if is_cav_arr[veh_idx] else 0.3
        ax.scatter(time_arr, pos[:, veh_idx], c=spd[:, veh_idx],
                   cmap='RdYlGn', s=0.3, vmin=0, vmax=10, alpha=alpha_val)

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Position (m)')
    ax.set_title(f'Space-Time Diagram \u2014 {mode_name.upper()} (HR=0.75)')
    plt.colorbar(ax.collections[0], ax=ax, label='Speed (m/s)')
    ax.axvline(x=3.0, color='red', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_DIR, f'spacetime_{mode_name}.png'), dpi=150)
    plt.show()

---
## 9. Export Results

In [ ]:
results_summary = {
    'total_timesteps': global_step,
    'training_time_min': total_time / 60,
    'fps_avg': np.mean(log_fps),
    'final_reward': log_rewards[-1] if log_rewards else None,
    'best_reward': max(log_rewards) if log_rewards else None,
    'sweep_results': sweep_results,
    'config': {
        'num_envs': NUM_ENVS,
        'rollout_steps': ROLLOUT_STEPS,
        'total_timesteps': TOTAL_TIMESTEPS,
        'lr_actor': LR_ACTOR,
        'lr_critic': LR_CRITIC,
        'hidden_dim': HIDDEN_DIM,
        'minibatch_size': MINIBATCH_SIZE,
        'ppo_epochs': PPO_EPOCHS,
        'gamma': GAMMA,
        'device': 'cpu',
        'num_threads': torch.get_num_threads(),
    },
}

with open(os.path.join(CHECKPOINT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(results_summary, f, indent=2)

log_df = pd.DataFrame({
    'step': log_steps,
    'reward': log_rewards,
    'pg_loss': log_pg_loss,
    'v_loss': log_v_loss,
    'entropy': log_entropy,
    'alpha_mean': log_alpha_mean,
    'fps': log_fps,
})
log_df.to_csv(os.path.join(CHECKPOINT_DIR, 'training_log.csv'), index=False)

print(f"All results saved to: {CHECKPOINT_DIR}")
for f_name in sorted(os.listdir(CHECKPOINT_DIR)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f_name))
    print(f"  {f_name:40s}  {size/1024:.1f} KB")

---
## 10. CPU Throughput Benchmark

In [ ]:
print(f'Benchmarking CPU environment throughput ({torch.get_num_threads()} threads)...')
bench_envs = [32, 64, 128, 256, 512]
bench_results = []

for n_envs in bench_envs:
    bench_env = VecRingRoadEnv(num_envs=n_envs, cfg=env_cfg, device=DEVICE)
    bench_env.reset()
    da = torch.zeros(n_envs, N_VEHICLES, device=DEVICE)

    # Warmup
    for _ in range(20):
        bench_env.step(da)

    n_steps = 200
    t0 = time.time()
    for _ in range(n_steps):
        bench_env.step(da)
    elapsed = time.time() - t0

    fps = (n_envs * n_steps) / elapsed
    bench_results.append({'num_envs': n_envs, 'fps': fps})
    print(f"  {n_envs:>5} envs: {fps:>10,.0f} env-steps/sec")
    del bench_env

if bench_results:
    fig, ax = plt.subplots(figsize=(8, 4))
    envs = [r['num_envs'] for r in bench_results]
    fps_vals = [r['fps'] for r in bench_results]
    ax.bar([str(e) for e in envs], fps_vals, color='teal', alpha=0.8)
    ax.set_xlabel('Number of Parallel Environments')
    ax.set_ylabel('Environment Steps / Second')
    ax.set_title(f'CPU Throughput ({torch.get_num_threads()} threads)')
    for i, v in enumerate(fps_vals):
        ax.text(i, v + max(fps_vals)*0.02, f'{v:,.0f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(CHECKPOINT_DIR, 'benchmark_cpu.png'), dpi=150)
    plt.show()